In [0]:
spark.conf.set(
    "fs.azure.account.key.stroadaccident01.dfs.core.windows.net",
    "<STORAGE_ACCOUNT_KEY>"
)

In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_path = "abfss://bronze@stroadaccident01.dfs.core.windows.net/"
silver_path = "abfss://silver@stroadaccident01.dfs.core.windows.net/"

tables = [
    "accident_details",
    "location_details",
    "weather_details",
    "road_features",
    "daylight_conditions"
]

In [0]:
from delta.tables import DeltaTable

# Read Bronze
df = spark.read.format("delta").load(f"{bronze_path}accident_details")

# -------------------------
# Data Quality Validations
# -------------------------
df = df.filter(col("ID").isNotNull())

df = df.dropDuplicates(["ID"])

df = df.filter(col("Severity").between(1, 4))

df = df.filter(col("Start_Time").isNotNull())

df = df.filter(col("End_Time") >= col("Start_Time"))

# -------------------------
# Silver Path
# -------------------------
silver_table_path = f"{silver_path}accident_details"

# -------------------------
# First Run -> Create Table
# Second Run -> UPSERT
# -------------------------
if not DeltaTable.isDeltaTable(spark, silver_table_path):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_table_path)
    )

    print("accident_details created in Silver")

else:

    delta_table = DeltaTable.forPath(spark, silver_table_path)

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            "target.ID = source.ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("accident_details UPSERT completed")

In [0]:
from delta.tables import DeltaTable

df = spark.read.format("delta").load(f"{bronze_path}location_details")

# Validations
df = df.filter(col("ID").isNotNull())
df = df.filter(col("State").isNotNull())
df = df.filter(col("City").isNotNull())
df = df.filter(col("Start_Lat").between(-90, 90))
df = df.filter(col("Start_Lng").between(-180, 180))

silver_table_path = f"{silver_path}location_details"

if not DeltaTable.isDeltaTable(spark, silver_table_path):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_table_path)
    )

    print("location_details created in Silver")

else:

    delta_table = DeltaTable.forPath(spark, silver_table_path)

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            "target.ID = source.ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("location_details UPSERT completed")

In [0]:
from delta.tables import DeltaTable

df = spark.read.format("delta").load(f"{bronze_path}weather_details")

# Validations
df = df.filter(col("Weather_Timestamp").isNotNull())
df = df.filter(col("Temperature_F_").isNotNull())
df = df.filter(col("Humidity___").between(0,100))
df = df.filter(col("Visibility_mi_") >= 0)
df = df.filter(col("Wind_Speed_mph_") >= 0)

silver_table_path = f"{silver_path}weather_details"

if not DeltaTable.isDeltaTable(spark, silver_table_path):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_table_path)
    )

    print("weather_details created in Silver")

else:

    delta_table = DeltaTable.forPath(spark, silver_table_path)

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            "target.ID = source.ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("weather_details UPSERT completed")

In [0]:
from delta.tables import DeltaTable

df = spark.read.format("delta").load(f"{bronze_path}road_features")

# Validations
df = df.filter(col("ID").isNotNull())
df = df.dropDuplicates(["ID"])
df = df.fillna(False)

silver_table_path = f"{silver_path}road_features"

if not DeltaTable.isDeltaTable(spark, silver_table_path):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_table_path)
    )

    print("road_features created in Silver")

else:

    delta_table = DeltaTable.forPath(spark, silver_table_path)

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            "target.ID = source.ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("road_features UPSERT completed")

In [0]:
from delta.tables import DeltaTable

df = spark.read.format("delta").load(f"{bronze_path}daylight_conditions")

# Validations
df = df.filter(col("ID").isNotNull())
df = df.filter(col("Sunrise_Sunset").isNotNull())
df = df.filter(col("Civil_Twilight").isNotNull())
df = df.filter(col("Nautical_Twilight").isNotNull())
df = df.filter(col("Astronomical_Twilight").isNotNull())

silver_table_path = f"{silver_path}daylight_conditions"

if not DeltaTable.isDeltaTable(spark, silver_table_path):

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_table_path)
    )

    print("daylight_conditions created in Silver")

else:

    delta_table = DeltaTable.forPath(spark, silver_table_path)

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            "target.ID = source.ID"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("daylight_conditions UPSERT completed")

In [0]:
for table in tables:

    df = spark.read.format("delta").load(f"{silver_path}{table}")

    print("="*60)
    print(table)
    print(df.count())

    display(df.limit(5))